# Memory and Message History

**What you will build:** multi-turn conversations in PydanticAI. The whole `WindowMemory`/`SummaryMemory` machinery you wrote in 0.6 becomes a single argument — but the underlying trade-offs don't disappear, and it's worth seeing why.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/14_memory_and_history.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.5 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## `message_history`: memory in one argument

Every run returns its messages via `result.all_messages()`. Feed them into the next run with `message_history=` and the agent remembers. That's the same append-the-list mechanic from 0.1, now handled for you.

In [ ]:
agent = Agent(model, system_prompt="You are a concise assistant.")

r1 = await agent.run("My name is Sam and I love jazz.")
print(r1.output)

r2 = await agent.run("What's my name, and what music do I like?",
                     message_history=r1.all_messages())
print(r2.output)

## A reusable chat loop

Wrap it so each turn carries the growing history forward automatically.

In [ ]:
agent = Agent(model, system_prompt="You are a friendly assistant with a good memory.")
history = []

async def chat(user_text):
    result = await agent.run(user_text, message_history=history)
    history[:] = result.all_messages()      # carry everything forward
    return result.output

print(await chat("I'm planning a trip to Japan in April."))
print(await chat("I especially love food and temples."))
print(await chat("Given all that, suggest one city for me."))   # uses both earlier turns

```{note}
PydanticAI makes memory convenient, not free. `history` still grows every turn, and it still has to fit the context window (0.0, 0.6). For long chats you apply the *same* strategies you built by hand — trim or summarize `history` before passing it back. Frameworks automate the plumbing, not the budgeting.
```

## The idiomatic budget: a history processor

You *could* trim `history` by hand before each run — but PydanticAI has a hook for exactly this: a **history processor**, a function that receives the message list right before it goes to the model and returns the version that should be sent. In 2.x it plugs in through the `capabilities` system. Your 0.6 `WindowMemory`, as a plug-in:

In [ ]:
from pydantic_ai.messages import ModelMessage
from pydantic_ai.capabilities import ProcessHistory

def keep_recent(messages: list[ModelMessage]) -> list[ModelMessage]:
    """Window strategy from 0.6: the first request (it carries the system prompt)
    plus the most recent messages."""
    if len(messages) <= 5:
        return messages
    return [messages[0]] + messages[-4:]

windowed = Agent(model, system_prompt="You are a concise assistant.",
                 capabilities=[ProcessHistory(keep_recent)])

history = []
for turn in ["My favourite colour is teal.",
             "I have a dog named Pixel.",
             "I live in Bilbao.",
             "My sister's name is June.",
             "What colour did I mention?"]:
    r = await windowed.run(turn, message_history=history)
    history[:] = r.all_messages()
    print(f"({len(history)} stored) {r.output[:80]}")

Watch the stored count **plateau** instead of growing — that's the window working. One subtlety worth knowing: the processor runs right before every model request, and the trimmed list is also what `all_messages()` hands back afterwards, so the trim is *real*, not cosmetic. If you want a full archive **as well as** a bounded window, keep your own append-only copy (or serialize before the trim — next section).

Two cautions for tool-using agents: never cut between a `tool-call` and its `tool-return` (providers reject the history), and always keep the message that carries the system prompt — both are why `keep_recent` slices whole messages from the front and pins `messages[0]`. A *summarizing* processor (0.6's other strategy) is this same hook with an LLM call inside.

## Sessions that survive the process: serialization

Everything so far dies with the kernel. For a real app you need to *store* a conversation and pick it up later — which is just serialization, and the messages are Pydantic objects, so they know how:

In [ ]:
from pathlib import Path
from pydantic_ai.messages import ModelMessagesTypeAdapter

# Save: bytes of JSON, straight to disk (or your database)
Path("session_sam.json").write_bytes(ModelMessagesTypeAdapter.dump_json(history))

# ... kernel restarts, new process, days later ...

restored = ModelMessagesTypeAdapter.validate_json(Path("session_sam.json").read_bytes())
r = await windowed.run("Remind me — what colour do I like?", message_history=restored)
print(r.output)

That save/load pair is the minimum viable "session store": key the file (or DB row) by a user id and you have multi-user persistent chat. It's also the exact problem LangGraph's **checkpointer** (2.3) solves as a first-class feature — automatically, per conversation thread, on every step. When you meet it, remember it's doing *this*, plus bookkeeping.

::::{dropdown} 🔍 What is inside `all_messages()`
:color: info

It's the structured record of the run: the system prompt, each user request, the model's responses, and any tool calls and their results — the typed version of the `messages` list you built by hand in Block 0. Print `len(history)` after a few turns to watch it grow; the serialize/restore pair you built above is what LangGraph's **checkpointer** (2.3) automates per conversation thread.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Prove it's the memory.** Remove `message_history=history` from `chat` and re-run the trip conversation. It forgets, exactly like the statelessness demo in 0.1.
2. **Break the window on purpose.** Instead of the `keep_recent` processor, slice crudely: `history[:] = result.all_messages()[-3:]`. Run several turns. **Done when:** you can name *both* things that go wrong with a crude slice — and why `keep_recent` avoids them.
3. **Two users, two sessions.** Using the serialization cell, keep `session_ana.json` and `session_ben.json`, alternate a few turns each, and confirm neither session leaks into the other. **Done when:** Ben's agent doesn't know Ana's name.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**2 — What goes wrong with `[-3:]`:** (a) the **system prompt vanishes** — it lives in the first `ModelRequest`, and your slice dropped it, so the agent silently loses its role and tone; (b) with tools in play a crude cut can **split a `tool-call` from its `tool-return`**, which providers reject as a malformed conversation (an HTTP 400, often cryptic). `keep_recent` keeps `messages[0]` for (a) and slices whole messages off the front for (b). Silent role loss is the nastier bug — nothing errors; the agent just gets blander.

**3 — Two sessions:**

```python
async def chat_as(user, text):
    path = Path(f"session_{user}.json")
    history = (ModelMessagesTypeAdapter.validate_json(path.read_bytes())
               if path.exists() else [])
    r = await agent.run(text, message_history=history)
    path.write_bytes(ModelMessagesTypeAdapter.dump_json(r.all_messages()))
    return r.output

print(await chat_as("ana", "Hi! I'm Ana and I'm learning cello."))
print(await chat_as("ben", "Hi! I'm Ben."))
print(await chat_as("ben", "What's my name? And do you know anyone learning an instrument?"))
```

Ben's agent knows Ben — and knows nothing about Ana, because her messages live in a file his session never loads. Isolation by construction. This per-user keying is exactly what `thread_id` does in LangGraph (2.3) and what the deployment chapter (3.0) must get right per request.
::::

**What's next:** in **1.5** we add **guardrails** — validating what goes in and what comes out, and defending against prompt injection.